# PyTorch 기초와 Keras 비교

**30일 딥러닝 완전 정복 · Day 15** — 강의 본문: [https://muy-bien-hwa.github.io/For-my-Claude/](https://muy-bien-hwa.github.io/For-my-Claude/)

### 이 노트북에서 할 일
- PyTorch 텐서와 GPU 사용법을 익힌다
- autograd 결과를 Day 10의 직접 만든 엔진과 비교한다
- nn.Module로 MLP를 정의하고 MNIST를 학습한다
- 같은 모델을 Keras로 학습해 비교한다

> ⚙️ **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 선택하세요. 셀은 위에서부터 `Shift+Enter`로 실행합니다.
> 그래프 라벨은 Colab 기본 글꼴 문제를 피하려고 영어로 표기했습니다.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
device = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.__version__, device)

## 1. 텐서

In [ ]:
x = torch.tensor([[1., 2.], [3., 4.]])
print(x.shape, x.dtype, x.device)
print(x @ x.T)
print(x.sum(dim=0), x.mean(dim=1, keepdim=True))
a = np.ones(3); t = torch.from_numpy(a); a[0] = 100; print("공유 메모리:", t)
z = torch.randn(8, 3, 32, 32)
print(z.view(8, -1).shape, z.permute(0, 2, 3, 1).shape, z.unsqueeze(0).shape)
g = torch.randn(1000, 1000, device=device)
print("GPU 행렬곱:", (g @ g).shape, (g @ g).device)

## 2. Autograd vs Day 10의 Value 엔진
Day 10에서 손으로 계산한 값과 같은지 확인하세요: `x.grad=-4, y.grad=-4, z.grad=3`.

In [ ]:
x = torch.tensor(-2.0, requires_grad=True); y = torch.tensor(5.0, requires_grad=True); z = torch.tensor(-4.0, requires_grad=True)
f = (x + y) * z
print(f.grad_fn)
f.backward()
print(x.grad, y.grad, z.grad)

# 뉴런 하나 + 손실 (Day 10 테스트 2와 같은 값이 나와야 함: dL/dw = -3.0551, dL/db = -2.0367)
x1 = torch.tensor(1.5); w1 = torch.tensor(-0.8, requires_grad=True); b = torch.tensor(0.5, requires_grad=True)
loss = (torch.tanh(w1 * x1 + b) - 1.0) ** 2
loss.backward()
print(f"loss {loss.item():.4f}  dL/dw {w1.grad.item():.4f}  dL/db {b.grad.item():.4f}")

# 누적 확인: zero_grad를 하지 않으면?
loss = (torch.tanh(w1 * x1 + b) - 1.0) ** 2; loss.backward()
print("두 번째 backward 후 (누적됨):", w1.grad.item())

## 3. nn.Module과 MNIST 학습

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_ds = datasets.MNIST("data", train=True, download=True, transform=tf)
test_ds = datasets.MNIST("data", train=False, download=True, transform=tf)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)
test_dl = DataLoader(test_ds, batch_size=512, num_workers=2)

class MLP(nn.Module):
    def __init__(self, d_in=784, d_hidden=256, n_classes=10, p_drop=0.2):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hidden); self.fc2 = nn.Linear(d_hidden, d_hidden); self.fc3 = nn.Linear(d_hidden, n_classes)
        self.drop = nn.Dropout(p_drop)
    def forward(self, x):
        x = x.flatten(1)
        x = self.drop(F.relu(self.fc1(x)))
        x = self.drop(F.relu(self.fc2(x)))
        return self.fc3(x)

model = MLP().to(device)
print(model)
print(sum(p.numel() for p in model.parameters()), "parameters")
for n, p in model.named_parameters(): print(n, tuple(p.shape))

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(5):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval(); correct = 0
    with torch.no_grad():
        for xb, yb in test_dl: correct += (model(xb.to(device)).argmax(1) == yb.to(device)).sum().item()
    print(f"epoch {epoch}  last batch loss {loss.item():.4f}  test acc {correct/len(test_ds):.4f}")

## 4. 흔한 실수 직접 체험
아래 셀은 **일부러 틀린** 코드입니다. 무엇이 문제인지 먼저 생각해 보고 실행하세요.

In [ ]:
# 실수 1: 모델 끝에 softmax를 붙이고 CrossEntropyLoss에 넣기
bad = nn.Sequential(MLP(), nn.Softmax(dim=1)).to(device)
opt_b = torch.optim.AdamW(bad.parameters(), lr=1e-3)
for i, (xb, yb) in enumerate(train_dl):
    loss = loss_fn(bad(xb.to(device)), yb.to(device)); opt_b.zero_grad(); loss.backward(); opt_b.step()
    if i == 300: break
print("softmax 두 번 적용 후 손실:", loss.item(), "← 확률(0~1)이 다시 softmax에 들어가므로 손실이 이론적으로 ln(e+9)-1 ≈ 1.46 아래로 내려갈 수 없다")

# 실수 2: 디바이스 불일치
try:
    model(torch.randn(2, 1, 28, 28))      # 모델은 GPU, 입력은 CPU
except RuntimeError as e:
    print("RuntimeError:", str(e)[:90])

## 5. 같은 모델을 Keras로
Colab에는 TensorFlow가 설치되어 있습니다. 코드 구조를 PyTorch 버전과 나란히 비교해 보세요.

In [ ]:
import tensorflow as tf
from tensorflow import keras
(x_tr, y_tr), (x_te, y_te) = keras.datasets.mnist.load_data()
x_tr, x_te = x_tr / 255.0, x_te / 255.0
kmodel = keras.Sequential([
    keras.layers.Input((28, 28)), keras.layers.Flatten(),
    keras.layers.Dense(256, activation="relu"), keras.layers.Dropout(0.2),
    keras.layers.Dense(256, activation="relu"), keras.layers.Dropout(0.2),
    keras.layers.Dense(10)])
kmodel.compile(optimizer=keras.optimizers.AdamW(1e-3),
               loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
kmodel.summary()
kmodel.fit(x_tr, y_tr, batch_size=128, epochs=5, validation_split=0.1)
print(kmodel.evaluate(x_te, y_te, verbose=0))

## 6. 과제
- PyTorch와 Keras 모델의 파라미터 수가 같은지 확인하세요.
- `MLP`를 `nn.Sequential` 한 줄 버전으로 다시 작성해 보세요.
- 학습 루프에서 `model.eval()`을 빼고 평가하면 정확도가 어떻게 달라지나요?